# Foco-LLM: teste fora do molde de dedução e rastreamento

Replicação pareada com 120 questões novas, gabaritos verificados simbolicamente e estruturas de frase diferentes. O treinamento usa somente os 80 exemplos originais; o conjunto de teste é congelado antes da inferência.


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"
if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError(f"Python 3.12 or 3.13 required; found {sys.version}")
if shutil.which("poetry") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "poetry==2.4.1"], check=True)
SOURCE_REF = "feat/deduction-specialist-7b"
repository = Path("/kaggle/temp/foco-llm-ood-chain")
output = Path("/kaggle/working/ood-chain-7b")
subprocess.run(
    [
        "git",
        "clone",
        "--branch",
        SOURCE_REF,
        "--single-branch",
        "https://github.com/Mathwesm/foco-llm.git",
        str(repository),
    ],
    check=True,
)
PINNED_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repository, text=True
).strip()
subprocess.run(
    ["poetry", "install", "--only", "main,inference", "--no-interaction"],
    cwd=repository,
    check=True,
)
print("Pinned source:", PINNED_COMMIT, flush=True)

In [ ]:
command = ["poetry", "run", "python", "scripts/run_deduction_specialist.py"]
subprocess.run([*command, "prepare", "--output", str(output)], cwd=repository, check=True)
training_manifest = json.loads((output / "dataset-manifest.json").read_text(encoding="utf-8"))
training_dataset = output / "training-dataset.json"
if (
    hashlib.sha256(training_dataset.read_bytes()).hexdigest()
    != training_manifest["training_sha256"]
):
    raise ValueError("Training dataset integrity mismatch")
ood_output = output / "ood-holdout"
subprocess.run(
    ["poetry", "run", "python", "scripts/run_ood_chain_holdout.py", "--output", str(ood_output)],
    cwd=repository,
    check=True,
)
ood_manifest = json.loads((ood_output / "manifest.json").read_text(encoding="utf-8"))
ood_dataset = ood_output / "dataset.json"
if hashlib.sha256(ood_dataset.read_bytes()).hexdigest() != ood_manifest["dataset_sha256"]:
    raise ValueError("OOD dataset integrity mismatch")
if (
    ood_manifest["dataset_sha256"]
    != "5a1ee80d4a2af8c27b50fc5e6fbde6eb2369388e01a21c298093447181d64951"
):
    raise ValueError("OOD dataset does not match preregistered hash")
subprocess.run(
    [
        "poetry",
        "run",
        "python",
        "scripts/run_cross_task_transfer.py",
        "evaluate",
        "--dataset",
        str(ood_dataset),
        "--output",
        str(output / "baseline"),
        "--backend",
        "cloud",
    ],
    cwd=repository,
    check=True,
)
print("Frozen OOD baseline complete", flush=True)

In [ ]:
subprocess.run(
    [*command, "train", "--output", str(output), "--checkpoint-every", "80"],
    cwd=repository,
    check=True,
)
completed = sorted((output / "training").glob("*/summary.json"))
if len(completed) != 1:
    raise RuntimeError(f"Expected one completed specialist run; found {len(completed)}")
adapter = completed[0].parent / "step-0320"
if not (adapter / "complete.json").is_file():
    raise RuntimeError("Step-0320 checkpoint is incomplete")
subprocess.run(
    [
        "poetry",
        "run",
        "python",
        "scripts/run_cross_task_transfer.py",
        "evaluate",
        "--dataset",
        str(ood_dataset),
        "--output",
        str(output / "specialist"),
        "--backend",
        "cloud",
        "--adapter",
        str(adapter),
        "--training-dataset",
        str(training_dataset),
    ],
    cwd=repository,
    check=True,
)


def read_one(root):
    """Load the sole summary for one evaluation arm."""
    paths = sorted(root.glob("*/summary.json"))
    if len(paths) != 1:
        raise RuntimeError(f"Expected one summary in {root}; found {len(paths)}")
    return json.loads(paths[0].read_text(encoding="utf-8"))


comparison = {
    "protocol": "ood-chain-holdout-7b-v1",
    "source_commit": PINNED_COMMIT,
    "training_manifest": training_manifest,
    "ood_manifest": ood_manifest,
    "training": json.loads(completed[0].read_text(encoding="utf-8")),
    "base": read_one(output / "baseline"),
    "deduction_specialist": read_one(output / "specialist"),
}
(output / "comparison.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
print(json.dumps(comparison, indent=2), flush=True)